In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../vehicles_us.csv")

df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


Hay 51,525 anuncios de vehículos.
Tiene 13 columnas.
Ocupa aproximadamente 7.7 MB de memoria.
price y days_listed son números enteros.
model_year, cylinders, odometer e is_4wd son numéricas con decimales, principalmente porque contienen valores faltantes.
Las demás columnas son texto.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


 is_4wd tiene 25,953 valores faltantes: aproximadamente la mitad de los anuncios no especifica si el vehículo tiene tracción 4x4.
paint_color tiene 9,267 valores faltantes.
odometer tiene 7,892 valores faltantes.
cylinders tiene 5,260 valores faltantes.
model_year tiene 3,619 valores faltantes.
Las demás columnas no tienen datos faltantes.
Para calcular el porcentaje de valores faltantes:

In [13]:
(car_data.isna().mean() * 100).sort_values(ascending=False).round(1)

is_4wd          50.4
paint_color     18.0
odometer        15.3
cylinders       10.2
model_year       7.0
condition        0.0
model            0.0
price            0.0
fuel             0.0
type             0.0
transmission     0.0
date_posted      0.0
days_listed      0.0
dtype: float64

is_4wd: falta en 50.4% de los anuncios. No conviene eliminar esas filas; muchas publicaciones no indican la tracción.
paint_color: falta en 18.0%.
odometer: falta en 15.3%. Es importante para análisis de precio, así que luego podemos rellenarlo con la mediana por modelo o excluir solo esos registros en gráficas donde sea necesario.
cylinders: falta en 10.2%.
model_year: falta en 7.0%.
Las demás variables están completas.

In [16]:
filtered_data = car_data[car_data["price"] <= 100000]

fig = px.histogram(
    filtered_data,
    x="price",
    nbins=60,
    title="Distribución de precios de vehículos"
)

fig.show()

La mayoría de los vehículos anunciados tiene precios inferiores a $20,000. La distribución presenta una cola hacia la derecha, causada por una cantidad reducida de vehículos de precio elevado.

In [19]:
fig = px.scatter(
    filtered_data,
    x="odometer",
    y="price",
    color="condition",
    title="Relación entre kilometraje, precio y condición"
)

fig.show()

Existe una tendencia negativa entre kilometraje y precio: los vehículos con mayor kilometraje suelen tener precios menores. Sin embargo, hay valores atípicos de kilometraje muy alto que dificultan visualizar el comportamiento de la mayoría de los anuncios. Puede haber una camioneta, auto clásico o vehículo de lujo con muchas millas y precio alto. Esos podrían ser los puntos aislados.

Los puntos muy alejados a la derecha, por encima de 600,000 millas, son valores poco comunes. Al forzar el eje hasta un millón de millas, comprimen visualmente en la gráfica, a la mayoría de vehículos, que se encuentran antes de 300,000 millas. Por eso propongo una segunda gráfica filtrada que permite ver con mayor claridad el patrón que representa a la mayoría de los anuncios.

In [24]:
scatter_data = filtered_data[
    filtered_data["odometer"] <= 300000
]

fig = px.scatter(
    scatter_data,
    x="odometer",
    y="price",
    color="condition",
    opacity=0.45,
    title="Relación entre kilometraje, precio y condición"
)

fig.show()

Con esto se puede afirmar mejor que los vehículos con poco kilometraje pueden tener una gran variedad de precios, pero los precios máximos disminuyen conforme aumenta el kilometraje. Los vehículos con alto kilometraje tienen principalmente precios bajos.

In [27]:
fig = px.box(
    filtered_data,
    x="type",
    y="price",
    title="Distribución de precios por tipo de vehículo",
    labels={
        "type": "Tipo de vehículo",
        "price": "Precio (USD)"
    }
)

fig.show()

pickup y truck tienen algunos de los precios típicos más altos.
sedan y hatchback tienen precios típicos más bajos.
SUV, coupe, convertible y offroad se ubican en rangos intermedios.
Todos los tipos muestran puntos altos aislados: son vehículos excepcionalmente caros dentro de su categoría.

El tipo de vehículo se relaciona con el precio. Las camionetas pickup y truck tienden a tener precios más altos, mientras que los sedanes y hatchbacks suelen concentrarse en precios más bajos.